# [Memory](https://langchain-ai.github.io/langgraph/concepts/memory/)
- LangGraph의 Memory는 AI가 대화 내용을 기억할 수 있게 해주는 기능입니다. 
- 사람이 대화할 때 이전에 말한 내용을 기억하는 것처럼, AI도 과거 대화를 기억해서 더 자연스러운 대화를 할 수 있게 해줍니다.


## 왜 Memory가 필요한가요?
> Memory가 없다면:
```text
    사용자: "내 이름은 김철수야"
    AI: "안녕하세요!"

    사용자: "내 이름이 뭐였지?"
    AI: "죄송해요, 모르겠습니다"
```
> Memory가 있다면:
```text
    사용자: "내 이름은 김철수야"
    AI: "안녕하세요 김철수님!"

    사용자: "내 이름이 뭐였지?"
    AI: "김철수님이라고 하셨어요!"
```


# LangGraph Memory 예시들

처음 배우는 학생들을 위한 단계별 예시를 준비했습니다!


## 1. Memory가 적용되지 않은 LangGraph 예시
**이름을 기억하지 못하는 챗봇**

### 1. AI 모델 설정

In [1]:
from langchain_ollama.chat_models import ChatOllama 

llm = ChatOllama(
    model="gemma3:4b",      # 이미 다운로드된 모델명 
    temperature=0.9,
    top_p=0.9,
    num_predict=512,
    keep_alive="10m"        # 로컬PC에서 모델이 메모리에 유지되는 시간 
)

### 2. 그래프 노드 정의 (Memory 없음)

In [2]:
from langgraph.graph import MessagesState

def chat_node(state: MessagesState):
    # 현재 메시지만 처리 (이전 대화 기억 안함)
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

### 3. 그래프 생성

In [3]:
from langgraph.graph import StateGraph, MessagesState, START, END

graph = StateGraph(MessagesState)
graph.add_node("chat", chat_node)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

### 4. 그래프 컴파일 (Memory 없음)

In [4]:
app_without_memory = graph.compile()

print("Memory 없는 챗봇이 준비되었습니다!")

Memory 없는 챗봇이 준비되었습니다!


### 5. 테스트 

In [5]:
from langchain_core.messages import HumanMessage

# 첫 번째 대화
print("사용자: '내 이름은 김철수야'")
result1 = app_without_memory.invoke({"messages": [HumanMessage(content="내 이름은 김철수야")]})
print(f"AI: {result1['messages'][-1].content}")


사용자: '내 이름은 김철수야'


/Users/gyoungwon-cho/dev/github/course_LLM/3. LangChain/1. colab/8. Memory/.venv/lib/python3.13/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=14eec952-a81f-444d-be68-804d8cc96550,id=14eec952-a81f-444d-be68-804d8cc96550
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=14eec952-a81f-444d-be68-804d8cc96550,id=8d0b3ddb-679d-4ce6-b9d7-237a4d990a57; trace=14eec952-a81f-444d-be68-804d8cc9

AI: 안녕하세요, 김철수님! 만나서 반갑습니다. 😊 

혹시 제가 김철수님께 무엇을 도와드릴까요? 궁금한 점이 있으시거나, 필요한 정보가 있으시면 언제든지 말씀해주세요.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=14eec952-a81f-444d-be68-804d8cc96550,id=d130037d-e038-4a16-b656-7963547ba9ff; trace=14eec952-a81f-444d-be68-804d8cc96550,id=233b87e0-5b31-4e6f-b712-08e439911a20; trace=14eec952-a81f-444d-be68-804d8cc96550,id=233b87e0-5b31-4e6f-b712-08e439911a20; trace=14eec952-a81f-444d-be68-804d8cc96550,id=00a632c7-0a7e-4546-9a48-075f1e2aad61; trace=14eec952-a81f-444d-be68-804d8cc96550,id=14eec952-a81f-444d-be68-804d8cc96550
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=7803b223-

In [6]:
# 두 번째 대화 (새로운 대화)
print("사용자: '내 이름이 뭐였지?'")
result2 = app_without_memory.invoke({"messages": [HumanMessage(content="내 이름이 뭐였지?")]})
print(f"AI: {result2['messages'][-1].content}")

print("\n결과: AI가 이전 대화를 기억하지 못해요!")

사용자: '내 이름이 뭐였지?'
AI: 죄송합니다. 저는 기억력이 없습니다. 질문자님의 이름을 알려주시겠어요?

결과: AI가 이전 대화를 기억하지 못해요!


## 2. Memory가 적용된 LangGraph 예시 (MemorySaver)
**이름을 기억하는 똑똑한 챗봇**


### 1. AI 모델 설정

In [7]:
from langchain_ollama.chat_models import ChatOllama 

llm = ChatOllama(
    model="gemma3:4b",      # 이미 다운로드된 모델명 
    temperature=0.9,
    top_p=0.9,
    num_predict=512,
    keep_alive="10m"        # 로컬PC에서 모델이 메모리에 유지되는 시간 
)

### 2. 그래프 노드 정의 (Memory 적용)

In [8]:
from langgraph.graph import MessagesState

def chat_node_with_memory(state: MessagesState):
    # 모든 이전 메시지들과 함께 처리 (Memory가 자동으로 관리)
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

### 3. 그래프 생성

In [9]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

graph_with_memory = StateGraph(MessagesState)
graph_with_memory.add_node("chat", chat_node_with_memory)
graph_with_memory.add_edge(START, "chat")
graph_with_memory.add_edge("chat", END)

### 4. Memory 저장소 설정 (여기가 핵심!)

In [10]:
memory = MemorySaver()

app_with_memory = graph_with_memory.compile(checkpointer=memory)

print("Memory가 적용된 챗봇이 준비되었습니다!")

Memory가 적용된 챗봇이 준비되었습니다!


### 5. 테스트 

In [11]:
from langchain_core.messages import HumanMessage
import uuid

# 중요: 같은 thread_id를 사용해야 대화가 연결됩니다!
memory_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": memory_id}}

print("=== Memory가 적용된 챗봇 테스트 ===")
print(f"config: {config}")

=== Memory가 적용된 챗봇 테스트 ===
config: {'configurable': {'thread_id': '238995c0-b33a-406a-8af7-ce6a08711f6f'}}


In [12]:
# 첫 번째 대화
print("사용자: '내 이름은 김철수야'")
result1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="내 이름은 김철수야")]}, 
    config=config  # 여기가 중요!
)
print(f"AI: {result1['messages'][-1].content}")

사용자: '내 이름은 김철수야'
AI: 안녕하세요, 김철수님! 만나서 반갑습니다. 😊



In [13]:
# 두 번째 대화 (같은 thread_id 사용)
print("사용자: '내 이름이 뭐였지?'")
result2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐였지?")]}, 
    config=config  # 같은 config 사용!
)
print(f"AI: {result2['messages'][-1].content}")

print("\n결과: AI가 이전 대화를 기억합니다!")

사용자: '내 이름이 뭐였지?'
AI: 당신의 이름은 김철수였습니다! 😉


결과: AI가 이전 대화를 기억합니다!
